# Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import balanced_accuracy_score, accuracy_score

# Load data

In [2]:
df = pd.read_csv('data/imdb/imdb_sentiment.csv')

# Store leaderboard
df_lb = df[df['split'] == 'leaderboard'].copy()
# Overwrite with labeled data only
df = df[df['split'] == 'labeled'].copy()

df.shape

(35000, 4)

In [3]:
df.head()

,id,split,review,sentiment
0,1,labeled,I really liked this Summerslam due to the look...,positive
1,2,labeled,Not many television shows appeal to quite as m...,positive
2,3,labeled,The film quickly gets to a major chase scene w...,negative
3,4,labeled,Jane Austen would definitely approve of this o...,positive
4,5,labeled,Expectations were somewhat high for me when I ...,negative


# Pre-trained Transformers

In [4]:
pip install transformers

   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.5 MB 6.3 MB/s eta 0:00:02
   ---------- ----------------------------- 2.9/11.5 MB 7.6 MB/s eta 0:00:02
   ----------------- ---------------------- 5.0/11.5 MB 8.9 MB/s eta 0:00:01
   ----------------------- ---------------- 6.8/11.5 MB 8.6 MB/s eta 0:00:01
   ------------------------------- -------- 9.2/11.5 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.5 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------- 11.5/11.5 MB 9.2 MB/s  0:00:01
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   -------------------------- ------------- 1.8/2.7 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 7.6 MB/s  0:00:00

   ------------- -------------------------- 1/3 [tokenizers]
   -------------------------- ------------- 2/3 [transformers]
   -------------------------- ------------- 2/3 

In [5]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [6]:
from transformers import pipeline

In [7]:
# This model by default predicts positive/negative

model = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    framework='pt'
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\fundacion\PycharmProjects\cifo_machine_learning_2026\.venv_py312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fundacion\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [8]:
# Take "raw" (minimum cleaning) reviews, without feature engineering

X = df['review']
y = df['sentiment'].map({'negative': 0, 'positive': 1})

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

In [9]:
# Prediction (may take a few minutes)

preds = model(X_test_raw.iloc[:10].values.tolist(), truncation=True)

In [10]:
# Predicted label + probability

preds

[{'label': 'NEGATIVE', 'score': 0.9994311928749084},
 {'label': 'POSITIVE', 'score': 0.9943444728851318},
 {'label': 'POSITIVE', 'score': 0.9966157078742981},
 {'label': 'NEGATIVE', 'score': 0.9005122184753418},
 {'label': 'NEGATIVE', 'score': 0.9995997548103333},
 {'label': 'NEGATIVE', 'score': 0.9998040795326233},
 {'label': 'POSITIVE', 'score': 0.9896134734153748},
 {'label': 'POSITIVE', 'score': 0.9996646642684937},
 {'label': 'POSITIVE', 'score': 0.994300127029419},
 {'label': 'POSITIVE', 'score': 0.993245542049408}]

In [11]:
y_test_raw.iloc[:10]

17813    0
6857     1
7672     1
9704     0
14303    0
26304    0
3202     1
27310    1
11215    1
20490    1
Name: sentiment, dtype: int64

In [12]:
# Convert labels to 0 or 1 and check test

y_distil_pred_test = pd.DataFrame(preds)['label'].map({'NEGATIVE': 0, 'POSITIVE': 1})
print('Test Accuracy:', accuracy_score(y_test_raw.iloc[:10], y_distil_pred_test))

Test Accuracy: 1.0
